In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# =========================
# Load Dataset
# =========================
df = pd.read_csv("failure_risk_dataset.csv")  # change path if needed

# Features and target
X = df[[
    "prompt_len",
    "code_len",
    "ast_nodes",
    "avg_complexity",
    "mean_logprob",
    "confidence"
]]

y = df["failed"]

# =========================
# Train Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# Pipeline
# =========================
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

# =========================
# Hyperparameter Grid
# =========================
param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear"]
}

# =========================
# Grid Search
# =========================
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid.best_params_)

# =========================
# Evaluation
# =========================
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best Parameters:
{'model__C': 1, 'model__penalty': 'l2', 'model__solver': 'liblinear'}

Accuracy: 0.7324561403508771

Confusion Matrix:
[[ 25  47]
 [ 14 142]]

Classification Report:
              precision    recall  f1-score   support

           0       0.64      0.35      0.45        72
           1       0.75      0.91      0.82       156

    accuracy                           0.73       228
   macro avg       0.70      0.63      0.64       228
weighted avg       0.72      0.73      0.71       228



In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = pd.read_csv("failure_risk_dataset.csv")

X = df[["prompt_len","code_len","ast_nodes","avg_complexity","mean_logprob","confidence"]]
y = df["failed"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

param_grid = {
    "n_estimators":[100,200,300],
    "max_depth":[None,5,10,20],
    "min_samples_split":[2,5,10],
    "min_samples_leaf":[1,2,4]
}

grid = GridSearchCV(
    RandomForestClassifier(),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train,y_train)

print("Best params:",grid.best_params_)

y_pred = grid.best_estimator_.predict(X_test)
print(classification_report(y_test,y_pred))

Best params: {'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
              precision    recall  f1-score   support

           0       0.85      0.69      0.76        72
           1       0.87      0.94      0.90       156

    accuracy                           0.86       228
   macro avg       0.86      0.82      0.83       228
weighted avg       0.86      0.86      0.86       228



In [6]:
!pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score
)
from xgboost import XGBClassifier

# Load dataset
df = pd.read_csv("failure_risk_dataset.csv")

X = df[[
    "prompt_len",
    "code_len",
    "ast_nodes",
    "avg_complexity",
    "mean_logprob",
    "confidence"
]]

y = df["failed"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# XGBoost model
model = XGBClassifier(
    eval_metric="logloss",
    use_label_encoder=False
)

# Hyperparameter tuning
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid.best_params_)

best_model = grid.best_estimator_

# Predictions
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\nAccuracy:", accuracy)
print("F1 Score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 48 candidates, totalling 240 fits

Best Parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}

Accuracy: 0.8333333333333334
F1 Score: 0.888235294117647
ROC-AUC: 0.9378116096866097

Confusion Matrix:
[[ 39  33]
 [  5 151]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.54      0.67        72
           1       0.82      0.97      0.89       156

    accuracy                           0.83       228
   macro avg       0.85      0.75      0.78       228
weighted avg       0.84      0.83      0.82       228



c:\Users\VINIL\Desktop\Failure_Aware_Agent\Environment\lib\site-packages\xgboost\training.py:200: UserWarning: [22:11:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score
)

# Load dataset
df = pd.read_csv("failure_risk_dataset.csv")

X = df[[
    "prompt_len",
    "code_len",
    "ast_nodes",
    "avg_complexity",
    "mean_logprob",
    "confidence"
]]

y = df["failed"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Model
model = GradientBoostingClassifier()

# Hyperparameter tuning
param_grid = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [3, 5],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("\nBest Parameters:")
print(grid.best_params_)

best_model = grid.best_estimator_

# Predictions
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\nAccuracy:", accuracy)
print("F1 Score:", f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 72 candidates, totalling 360 fits

Best Parameters:
{'learning_rate': 0.01, 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}

Accuracy: 0.8333333333333334
F1 Score: 0.8869047619047619
ROC-AUC: 0.9456463675213674

Confusion Matrix:
[[ 41  31]
 [  7 149]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.57      0.68        72
           1       0.83      0.96      0.89       156

    accuracy                           0.83       228
   macro avg       0.84      0.76      0.79       228
weighted avg       0.84      0.83      0.82       228



In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

# Load dataset
df = pd.read_csv("failure_risk_dataset.csv")

X = df[["prompt_len","code_len","ast_nodes","avg_complexity","mean_logprob","confidence"]].values
y = df["failed"].values

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling (important for ANN)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

# ANN model
class ANN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Hyperparameter search
param_grid = {
    "hidden_size":[8,16,32],
    "lr":[0.001,0.01],
    "epochs":[100,200]
}

best_f1 = 0
best_model = None

for params in ParameterGrid(param_grid):

    model = ANN(6, params["hidden_size"])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=params["lr"])

    for epoch in range(params["epochs"]):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        probs = model(X_test).numpy().flatten()
        preds = (probs > 0.5).astype(int)

        f1 = f1_score(y_test.numpy(), preds)

        if f1 > best_f1:
            best_f1 = f1
            best_model = model
            best_params = params

print("Best Params:", best_params)

# Final evaluation
with torch.no_grad():
    probs = best_model(X_test).numpy().flatten()
    preds = (probs > 0.5).astype(int)

accuracy = accuracy_score(y_test.numpy(), preds)
roc_auc = roc_auc_score(y_test.numpy(), probs)

print("\nAccuracy:", accuracy)
print("F1:", best_f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test.numpy(), preds))

print("\nClassification Report:")
print(classification_report(y_test.numpy(), preds))

Best Params: {'epochs': 200, 'hidden_size': 32, 'lr': 0.01}

Accuracy: 0.8728070175438597
F1: 0.9096573208722741
ROC-AUC: 0.9396367521367521

Confusion Matrix:
[[ 53  19]
 [ 10 146]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.84      0.74      0.79        72
         1.0       0.88      0.94      0.91       156

    accuracy                           0.87       228
   macro avg       0.86      0.84      0.85       228
weighted avg       0.87      0.87      0.87       228



In [2]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# =========================
# Load dataset
# =========================
df = pd.read_csv("failure_risk_dataset.csv")

X = df[["prompt_len","code_len","ast_nodes","avg_complexity","mean_logprob","confidence"]].values
y = df["failed"].values

# =========================
# Train Test Split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# Feature Scaling
# =========================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

# =========================
# ANN Model
# =========================
class ANN(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()

        layers = []
        in_features = input_size

        for _ in range(num_layers):
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_features = hidden_size

        layers.append(nn.Linear(hidden_size,1))
        layers.append(nn.Sigmoid())

        self.net = nn.Sequential(*layers)

    def forward(self,x):
        return self.net(x)

# =========================
# Hyperparameter Space
# =========================
param_grid = {
    "hidden_size":[4,8,16,32,64],
    "num_layers":[1,2,3],
    "lr":[0.0001,0.0005,0.001,0.005,0.01],
    "batch_size":[16,32,64],
    "epochs":[100,200,300],
    "dropout":[0.0,0.2,0.4]
}

# Random search (much faster)
param_list = list(ParameterSampler(param_grid, n_iter=40, random_state=42))

best_f1 = 0
best_model = None
best_params = None

# =========================
# Training Loop
# =========================
for params in param_list:

    model = ANN(
        input_size=6,
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"]
    )

    criterion = nn.BCELoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=params["lr"]
    )

    dataset = TensorDataset(X_train, y_train)

    loader = DataLoader(
        dataset,
        batch_size=params["batch_size"],
        shuffle=True
    )

    # Training
    for epoch in range(params["epochs"]):

        for xb, yb in loader:

            optimizer.zero_grad()

            preds = model(xb)

            loss = criterion(preds, yb)

            loss.backward()

            optimizer.step()

    # Evaluation
    with torch.no_grad():

        probs = model(X_test).numpy().flatten()

        preds = (probs > 0.5).astype(int)

        f1 = f1_score(y_test.numpy(), preds)

        if f1 > best_f1:

            best_f1 = f1
            best_model = model
            best_params = params

# =========================
# Best Model Results
# =========================
print("\nBest Params:")
print(best_params)

with torch.no_grad():

    probs = best_model(X_test).numpy().flatten()

    preds = (probs > 0.5).astype(int)

accuracy = accuracy_score(y_test.numpy(), preds)

roc_auc = roc_auc_score(y_test.numpy(), probs)

print("\nAccuracy:", accuracy)
print("F1 Score:", best_f1)
print("ROC-AUC:", roc_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test.numpy(), preds))

print("\nClassification Report:")
print(classification_report(y_test.numpy(), preds))


Best Params:
{'num_layers': 1, 'lr': 0.001, 'hidden_size': 64, 'epochs': 200, 'dropout': 0.2, 'batch_size': 16}

Accuracy: 0.8771929824561403
F1 Score: 0.910828025477707
ROC-AUC: 0.9403490028490029

Confusion Matrix:
[[ 56  16]
 [ 12 144]]

Classification Report:
              precision    recall  f1-score   support

         0.0       0.82      0.78      0.80        72
         1.0       0.90      0.92      0.91       156

    accuracy                           0.88       228
   macro avg       0.86      0.85      0.86       228
weighted avg       0.88      0.88      0.88       228

